# 13 文本大模型部署面试检查清单

目标：把前面学到的 tokenizer、dtype、量化、LoRA、KV cache、batching、服务化指标串成面试表达。


## 1. 安装依赖


In [ ]:
from pathlib import Path

base = Path.cwd()
requirements_path = base / "requirements.txt"
advanced_requirements_path = base / "advanced" / "requirements-advanced.txt"

if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

if not advanced_requirements_path.exists():
    if Path("requirements-advanced.txt").exists():
        advanced_requirements_path = Path("requirements-advanced.txt")
    else:
        advanced_requirements_path = Path("../advanced/requirements-advanced.txt")

print("requirements:", requirements_path)
print("advanced requirements:", advanced_requirements_path)
%pip install -r {requirements_path} -r {advanced_requirements_path}


## 2. 模型与工具函数


In [ ]:
import gc
import os
from pathlib import Path
from time import perf_counter

import torch
from modelscope import snapshot_download
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE == "modelscope":
        return snapshot_download(model_id)
    return model_id


def cleanup(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def cuda_memory(label=""):
    if not torch.cuda.is_available():
        print(label, "cuda unavailable")
        return
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"{label} allocated={allocated:.2f}GB reserved={reserved:.2f}GB peak={peak:.2f}GB")


MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)
print("cuda =", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu =", torch.cuda.get_device_name(0))
    print("bf16 supported =", torch.cuda.is_bf16_supported())


## 3. 一段代码快速读懂模型配置


In [ ]:
config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
keys = [
    "model_type",
    "hidden_size",
    "num_hidden_layers",
    "num_attention_heads",
    "num_key_value_heads",
    "vocab_size",
    "max_position_embeddings",
]
for key in keys:
    print(f"{key}:", getattr(config, key, "unknown"))

attention_heads = config.num_attention_heads
kv_heads = getattr(config, "num_key_value_heads", attention_heads)
print("GQA group size:", attention_heads // kv_heads if kv_heads else "unknown")


## 4. 面试高频 1：一次请求从文本到输出经历什么？

标准链路：raw text -> chat template -> tokenizer -> input_ids/attention_mask -> prefill -> KV cache -> decode loop -> logits -> sampling/greedy -> decode text。


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
messages = [
    {"role": "system", "content": "你是一个面试官。"},
    {"role": "user", "content": "请解释 tokenizer 的作用。"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
encoded = tokenizer(prompt, return_tensors="pt")
print(prompt)
print("input_ids shape:", encoded["input_ids"].shape)
print("first 20 ids:", encoded["input_ids"][0, :20].tolist())


## 5. 面试高频 2：显存怎么估算？

部署显存不是只看权重，还要看 KV cache、activation、batch、并发、框架开销。


In [ ]:
def memory_table(num_params_m=500, seq_len=2048, batch=1):
    num_params = num_params_m * 1_000_000
    print("weight memory:")
    for name, b in [("fp32", 4), ("fp16/bf16", 2), ("int8", 1), ("int4", 0.5)]:
        print(f"  {name:<10} {num_params*b/1024**3:.2f}GB")

    layers = getattr(config, "num_hidden_layers", 0)
    heads = getattr(config, "num_attention_heads", 1)
    kv_heads = getattr(config, "num_key_value_heads", heads)
    head_dim = getattr(config, "hidden_size", 0) // heads
    kv_bytes = layers * batch * seq_len * kv_heads * head_dim * 2 * 2
    print("kv cache fp16/bf16:", f"{kv_bytes/1024**3:.3f}GB")

memory_table(num_params_m=500, seq_len=2048, batch=1)


## 6. 面试高频 3：优化手段怎么选？


In [ ]:
optimizations = [
    ("FP16/BF16", "显存约减半，兼容好，优先使用"),
    ("INT8/INT4 量化", "显存更低，依赖量化 kernel，可能有精度和速度 tradeoff"),
    ("LoRA", "训练时只更新 adapter，适合低成本领域/格式适配"),
    ("KV cache", "降低 decode 重复计算，但长上下文/高并发时占显存"),
    ("batching", "提升吞吐，但可能增加单请求延迟"),
    ("streaming", "不减少总计算量，但降低首屏等待"),
    ("vLLM/TGI", "生产服务框架，支持连续批处理、PagedAttention、OpenAI API 等"),
]
for name, note in optimizations:
    print(f"{name}: {note}")


## 7. 可以直接背的回答模板

**量化**：量化把权重从 FP16/FP32 映射到 INT8/INT4，并保存 scale/zero-point 或 group-wise scale。推理时通过专门 kernel 分块反量化或边反量化边 matmul，降低权重常驻显存，但可能带来精度损失和兼容性问题。

**LoRA 微调**：LoRA 冻结原模型，在 attention/MLP 线性层旁训练低秩矩阵，训练参数少、显存低、保存的是 adapter。适合风格、格式和领域任务适配。

**KV cache**：KV cache 保存历史 token 在每层 attention 中的 key/value，decode 时避免重复计算历史上下文。它随 batch、seq_len、layers、kv_heads、head_dim 线性增长。

**prefill vs decode**：prefill 一次性处理 prompt，建立 KV cache；decode 自回归逐 token 生成，每步读取 KV cache 并追加新 KV。
